<a href="https://colab.research.google.com/github/miso88oup/image-caption-generator/blob/main/notebooks/05_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Caption Generator — Demo

Web interface: upload an image, choose the CNN encoder and decoding method, and watch the caption being generated word by word.

## 1. Setup

In [4]:
!pip install -q gradio

In [3]:
import os
import json
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import gradio as gr
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.applications import vgg16, resnet50, inception_v3
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/image-caption-project'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')

with open(os.path.join(DATA_DIR, 'vocab.json')) as f:
    vocab = json.load(f)
with open(os.path.join(DATA_DIR, 'splits.json')) as f:
    splits = json.load(f)

word2idx = vocab['word2idx']
idx2word = {i: w for w, i in word2idx.items()}
max_length = vocab['max_length']
PAD, UNK = 0, 1
START, END = word2idx['startseq'], word2idx['endseq']

with open(os.path.join(PROJECT_DIR, 'results', 'best_seeds.json')) as f:
    best_seeds = json.load(f)

Mounted at /content/drive


## 2. Load Models

In [5]:
def build_encoder(name):
    if name == 'vgg16':
        base = vgg16.VGG16(weights='imagenet')
        model = Model(base.input, base.get_layer('fc2').output)
        return model, vgg16.preprocess_input, (224, 224)

    if name == 'resnet50':
        model = resnet50.ResNet50(weights='imagenet', include_top=False, pooling='avg')
        return model, resnet50.preprocess_input, (224, 224)

    if name == 'inceptionv3':
        model = inception_v3.InceptionV3(weights='imagenet', include_top=False, pooling='avg')
        return model, inception_v3.preprocess_input, (299, 299)


ENCODERS = {'VGG16': 'vgg16', 'ResNet50': 'resnet50', 'InceptionV3': 'inceptionv3'}

encoders = {name: build_encoder(name) for name in ENCODERS.values()}
decoders = {name: keras.models.load_model(os.path.join(MODELS_DIR, f'{name}_seed{best_seeds[name]}.keras'))
            for name in ENCODERS.values()}
print('Models loaded')

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Models loaded


## 3. Caption Generation

In [6]:
def extract_features(name, image):
    model, preprocess, size = encoders[name]
    resized = tf.image.resize(np.array(image.convert('RGB')), size).numpy()
    feature = model.predict(preprocess(resized[None].copy()), verbose=0)[0]
    return feature, resized.astype('uint8')


def next_word_probs(model, feats, seqs):
    probs = model([feats, seqs], training=False)[:, -1, :].numpy()
    probs[:, [PAD, UNK, START]] = 0.0
    return probs


def greedy_steps(model, feature):
    seq, steps = [START], []
    for _ in range(max_length - 1):
        probs = next_word_probs(model, feature[None], np.array([seq]))[0]
        top5 = np.argsort(probs)[::-1][:5]
        chosen = int(top5[0])
        steps.append(([(idx2word[int(i)], float(probs[i])) for i in top5], idx2word[chosen]))
        if chosen == END:
            break
        seq.append(chosen)
    return steps


def beam_candidates(model, feature, k=3):
    seqs = np.full((k, 1), START)
    scores = np.full(k, -np.inf)
    scores[0] = 0.0
    finished = np.zeros(k, dtype=bool)

    for _ in range(max_length - 1):
        log_probs = np.log(next_word_probs(model, np.repeat(feature[None], k, axis=0), seqs) + 1e-12)
        vocab_size = log_probs.shape[1]
        log_probs[finished] = -np.inf
        log_probs[:, PAD] = np.where(finished, 0.0, -np.inf)

        total = (scores[:, None] + log_probs).ravel()
        top = np.argsort(-total)[:k]
        beam_idx, word_idx = top // vocab_size, top % vocab_size

        scores = total[top]
        seqs = np.concatenate([seqs[beam_idx], word_idx[:, None]], axis=1)
        finished = finished[beam_idx] | (word_idx == END)
        if finished.all():
            break

    results = []
    for seq, score in zip(seqs, scores):
        words = [idx2word[int(i)] for i in seq[1:] if i not in (END, PAD)]
        length = max(int((seq[1:] != PAD).sum()), 1)
        results.append((' '.join(words), round(float(score / length), 3)))
    return sorted(results, key=lambda r: r[1], reverse=True)


def feature_plot(feature, name):
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.plot(feature, linewidth=0.6)
    ax.set_title(f'{name} feature vector ({len(feature)} values)')
    ax.set_xlabel('Feature index')
    plt.tight_layout()
    plt.close(fig)
    return fig

## 4. Interface

In [7]:
STEP_COLUMNS = ['Step', 'Top-5 candidates (probability)', 'Chosen word']
BEAM_COLUMNS = ['Candidate caption', 'Score']


def run(image, encoder_label, method):
    if image is None:
        raise gr.Error('Please upload an image.')

    name = ENCODERS[encoder_label]
    feature, resized = extract_features(name, image)
    plot = feature_plot(feature, encoder_label)
    model = decoders[name]

    steps = greedy_steps(model, feature)
    rows, words = [], []
    for t, (top5, chosen) in enumerate(steps, start=1):
        rows.append([t, ', '.join(f'{w} ({p:.2f})' for w, p in top5), chosen])
        if chosen != 'endseq':
            words.append(chosen)
        if method == 'Greedy':
            yield ' '.join(words), resized, plot, pd.DataFrame(rows, columns=STEP_COLUMNS), pd.DataFrame(columns=BEAM_COLUMNS)
            time.sleep(0.35)

    if method != 'Greedy':
        beams = beam_candidates(model, feature, k=int(method[-2]))
        yield beams[0][0], resized, plot, pd.DataFrame(rows, columns=STEP_COLUMNS), pd.DataFrame(beams, columns=BEAM_COLUMNS)

In [ ]:
import kagglehub
IMAGE_DIR = os.path.join(kagglehub.dataset_download('adityajn105/flickr8k'), 'Images')
examples = [os.path.join(IMAGE_DIR, img) for img in splits['test'][:6]]

with gr.Blocks(title='Image Caption Generator') as demo:
    gr.Markdown('# Image Caption Generator\nCNN encoder + LSTM decoder trained on Flickr8k')

    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type='pil', label='Image')
            encoder_in = gr.Dropdown(list(ENCODERS), value='ResNet50', label='CNN encoder')
            method_in = gr.Radio(['Greedy', 'Beam search (k=3)', 'Beam search (k=5)'], value='Greedy', label='Decoding')
            button = gr.Button('Generate caption', variant='primary')
            gr.Examples(examples, inputs=image_in)

        with gr.Column():
            caption_out = gr.Textbox(label='Generated caption', lines=2)
            steps_out = gr.Dataframe(headers=STEP_COLUMNS, label='Word-by-word generation (greedy)')
            beams_out = gr.Dataframe(headers=BEAM_COLUMNS, label='Beam search candidates')

    with gr.Row():
        resized_out = gr.Image(label='CNN input (resized)')
        feature_out = gr.Plot(label='Extracted features')

    button.click(run, [image_in, encoder_in, method_in],
                 [caption_out, resized_out, feature_out, steps_out, beams_out])

demo.launch(share=True, debug=True)

Using Colab cache for faster access to the 'flickr8k' dataset.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8333d714932fd06764.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
